# 👂 OPERAÇÃO VOZ DO CLIENTE
### Desafio 3 · Trilha de Linguagem Natural · Olimpíada de IA Aplicada 2026

---

> **"Eu consigo ler umas duzentas avaliações por dia. Chegam quarenta mil. A conta não
> fecha, e o cliente que estava realmente irritado é justamente o que fica na parte da
> pilha que eu nunca alcanço."**
>
> — Vera Nogueira, Ouvidoria

---

<img src="https://lh3.googleusercontent.com/d/1c1m0E6nQcJi6bacgYfcMSA-AU_b3QrCN=w1600" width="100%"
     alt="Sala de ouvidoria à noite: Vera, de headset e blusa clara, olha para uma tela enquanto uma pilha altíssima de avaliações com estrelas flutua à sua frente. As do topo brilham em ciano e magenta; a base da pilha se perde no escuro, nunca lida.">


## 📨 Transmissão recebida · Prioridade máxima

*Três da tarde. O canal da Ouvidoria abre uma chamada para a sua equipe.*

---

**De:** Vera Nogueira · Ouvidoria e Pós-venda
**Para:** Equipe de Investigação em Linguagem
**Classificação:** urgente

> Coordeno a ouvidoria de um marketplace há nove anos. Vou expor o problema técnico com
> precisão, porque a formulação importa mais que a solução.
>
> Recebemos dezenas de milhares de avaliações de produto por dia, escritas em texto livre
> por clientes reais. Dispomos de um acervo grande já rotulado: cada avaliação marcada como
> vinda de um cliente **satisfeito** ou **insatisfeito**. A leitura manual não escala, e é
> isso que motiva a automação.
>
> O primeiro ponto não trivial é de onde veio esse rótulo. **Ele não foi atribuído por um
> especialista.** Ele foi derivado da nota em estrelas que o próprio cliente deu ao produto.
> Isso significa que a nossa "verdade" carrega o ruído da pessoa que avaliou: existe cliente
> que dá uma estrela porque a encomenda atrasou, e escreve, no mesmo texto, que o produto é
> excelente. Esse caso entra no acervo como insatisfeito. Vocês vão treinar um modelo em
> cima disso, e precisam saber disso desde já.
>
> O segundo ponto é por que precisamos de um modelo, se a estrela já existe. Porque o mesmo
> texto chega até nós por canais **onde não existe estrela nenhuma**: o chat do atendimento,
> o e-mail, o formulário de contato, as menções em rede social. O classificador é treinado
> nas avaliações porque é ali que existe rótulo, e depois é aplicado onde não existe.
>
> Some-se a isso a sobreposição de superfície entre as classes. Cliente satisfeito também
> reclama do prazo, também escreve em caixa alta, também usa ponto de exclamação. Os
> marcadores óbvios, portanto, aparecem nos dois lados. O sinal discriminante está em outro
> lugar, e parte do trabalho de vocês é localizá-lo.
>
> Uma observação sobre a assimetria de custo, que vai orientar a avaliação: marcar como
> insatisfeito quem estava satisfeito gera um contato desnecessário e ocupa a fila de
> atendimento. Deixar passar um cliente insatisfeito significa que ninguém o procura, e a
> próxima notícia que temos dele costuma ser uma reclamação pública ou a perda da conta.
> **As duas taxas de erro não são intercambiáveis**, e otimizar o agregado sem considerar
> isso produz um sistema mal calibrado para o uso real.
>
> — Vera Nogueira

---

*O acervo tem cerca de cento e dez mil avaliações rotuladas. O prazo é de uma semana.*


## 🎯 A sua missão

Construir um modelo de IA que lê uma avaliação escrita por um cliente e decide:
**quem escreveu estava SATISFEITO ou INSATISFEITO?**

### Representação de texto: o modelo vetorial

Texto é dado não estruturado de comprimento variável, e classificadores operam sobre
vetores numéricos de dimensão fixa. É necessário, portanto, um esquema de representação.

O adotado aqui é o **TF-IDF** (*term frequency–inverse document frequency*), que atribui a
cada termo um peso composto por dois fatores:

- **TF**, a frequência do termo no documento. Quanto mais vezes um termo aparece em uma
  avaliação, mais ele a caracteriza.
- **IDF**, o inverso da frequência do termo no corpus. Termos presentes em quase todos os
  documentos recebem peso próximo de zero; termos concentrados em poucos documentos recebem
  peso alto.

O produto dos dois implementa uma intuição precisa: **um termo é informativo quando é
frequente em um documento e raro no corpus**. Preposições e artigos são penalizados
automaticamente pelo IDF, sem necessidade de lista manual.

O resultado é uma matriz em que cada linha é uma avaliação e cada coluna um termo do
vocabulário. Ela é esparsa, ou seja, quase todos os valores são zero, já que cada avaliação
contém uma fração mínima do vocabulário total. Estruturas esparsas são armazenadas de forma
otimizada, o que torna o método viável mesmo com vocabulários grandes.

Feita a vetorização, o problema volta a ser tabular e a interface de sempre se aplica.

### 🗺️ O mapa da operação

| Bloco | O que vocês fazem | Como |
|:---|:---|:---|
| **1 · EXPLORAR** | Ler avaliações reais e caçar o que separa quem gostou de quem não gostou | Só clicando |
| **2 · CONFIGURAR** | Decidir como o texto vira número | Só arrastando |
| **3 · MODELAR** | Traduzir, treinar e prever | ✏️ Completando **11 lacunas** |
| **4 · AVALIAR** | Descobrir quais palavras entregam um cliente irritado | Só clicando |

> **Nota sobre o Bloco 3.** Das onze lacunas, **duas envolvem métodos de nome semelhante e
> semântica distinta**. A confusão entre elas produz um erro que não gera exceção, executa
> silenciosamente e infla o resultado medido. É um dos erros mais frequentes na prática
> profissional da área, e a verificação do bloco detecta exatamente esse caso.

> ⚠️ **Antes de começar:** vá em `Ambiente de execução` → `Alterar o tipo de ambiente de
> execução` e confirme que está em **CPU**. Esta operação inteira roda em segundos, sem
> precisar de placa de vídeo. É assim que garantimos que todas as equipes do Brasil
> compitam em pé de igualdade.
>
> Use `Arquivo` → `Salvar uma cópia no Drive` antes de começar, para não perder o trabalho.

In [ ]:
#@title 🛠️ Preparação: abrindo o acervo da Ouvidoria { display-mode: "form" }
# Endereços oficiais dos dados. A organização republica aqui se precisar trocar.
URL_TREINO = "https://drive.google.com/uc?export=download&id=13gKaLX98JaZOOq4BqdnjsxPEnmxXe6ux"
URL_TESTE = "https://drive.google.com/uc?export=download&id=1XBWfEtoujz7p_2E6nW-XEdn29xuXQKiT"

import pathlib, warnings, re
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

NOME_CLASSE = {0: "INSATISFEITO", 1: "SATISFEITO"}
# A classe que custa caro errar. Toda a avaliação do Bloco 4 gira em torno dela.
CLASSE_CRITICA = 0


def _carregar(url, nome_local):
    """Baixa do endereço oficial; se a rede falhar, usa cópia local ou pede envio."""
    print(f"Baixando {nome_local}...")
    try:
        return pd.read_csv(url)
    except Exception as erro:
        print(f"  não deu certo ({erro.__class__.__name__}).")
        if not pathlib.Path(nome_local).exists():
            from google.colab import files
            print(f"  envie o arquivo {nome_local}:")
            files.upload()
        return pd.read_csv(nome_local)


# Conjunto PÚBLICO: rotulado. Serve para treinar e validar.
dados = _carregar(URL_TREINO, "treino.csv")

# Conjunto de TESTE: tamanho fixo, rótulos ocultos. Base da comparação oficial.
_teste = _carregar(URL_TESTE, "teste.csv")
mensagens_teste = _teste["mensagem"]
# Os ids saem do próprio arquivo: recriá-los com arange desalinharia a submissão
# do gabarito se a numeração original mudasse.
IDS_TESTE = _teste["id_mensagem"].to_numpy()
N_TESTE = len(mensagens_teste)

_p_ins = (dados.rotulo == 0).mean() * 100
print("\nAmbiente carregado.\n")
print(f"  Acervo público (rotulado) ......... {len(dados):>6} avaliações")
print(f"     insatisfeitos (rótulo 0) ....... {(dados.rotulo==0).sum():>6}   {_p_ins:4.1f}%")
print(f"     satisfeitos   (rótulo 1) ....... {(dados.rotulo==1).sum():>6}   {100-_p_ins:4.1f}%")
print(f"  Conjunto de teste (sem rótulo) .... {N_TESTE:>6} avaliações  [tamanho fixo]")
print(f"\n  As classes NÃO estão equilibradas. Guardem esse número: ele volta no Bloco 4.")
print(f"  A submissão precisará ter exatamente {N_TESTE} linhas.\n")
for _, r in dados.head(2).iterrows():
    print(f"  [{NOME_CLASSE[r.rotulo]}]  {r.mensagem[:100]}...")

---
# 🔍 BLOCO 1 · EXPLORAR
## *Entrem na cabeça de quem escreveu*

> **Vera:** *"Leiam o acervo antes de modelar. Analista que não conhece o dado escolhe mal
> os parâmetros e não sabe interpretar o que o modelo aprendeu."*

Inspeção manual de amostras é etapa padrão antes da modelagem em processamento de
linguagem natural. O objetivo aqui é formular hipóteses sobre quais marcadores linguísticos
discriminam as classes, para depois confrontá-las com o que o modelo efetivamente aprender.

Enquanto leem, prestem atenção em três coisas que o dado tem de sobra:

- **Avaliações mistas.** *"Demorou na entrega. Tirando isso, a tv é excelente."* Existe
  elogio e existe reclamação no mesmo texto. Um dos dois define o rótulo. Qual?
- **Reclamação de logística × reclamação do produto.** São insatisfações diferentes, com
  vocabulário diferente, e recebem o mesmo rótulo.
- **Negação.** `recomendo` e `não recomendo` compartilham a palavra que mais pesa. Guardem
  isso: no Bloco 2 existe um parâmetro que trata exatamente disso.


In [ ]:
#@title 📬 Leia avaliações do acervo { display-mode: "form" }
#@markdown Escolha o tipo de avaliação e rode a célula (▶). Leia com atenção de investigador.
tipo = "insatisfeito"  #@param ["insatisfeito", "satisfeito"]
quantidade = 6  #@param {type:"slider", min:3, max:12, step:1}

alvo = 0 if tipo == "insatisfeito" else 1
amostra = dados[dados.rotulo == alvo].sample(quantidade, random_state=7)

print("="*70)
print(f"   {NOME_CLASSE[alvo]}  ·  {quantidade} avaliações do acervo")
print("="*70)
for k, (_, r) in enumerate(amostra.iterrows(), 1):
    print(f"\n[{k}] {r.mensagem}")
print("\n" + "="*70)
print("\nHIPÓTESES A FORMULAR:")
print("   A reclamação é do produto, da entrega ou do atendimento?")
print("   Quantas destas avaliações elogiam e reclamam ao mesmo tempo?")
print("   Que marcadores de emoção aparecem, e são exclusivos de uma das classes?")
print("   Alternem entre as classes e identifiquem o que efetivamente NÃO se sobrepõe.")
print("   Marcadores presentes nos dois lados têm baixo poder discriminante.")

In [ ]:
#@title 📊 Investigue os dados: escolha a variável E o tipo de gráfico { display-mode: "form" }
#@markdown Escolha a variável e o gráfico, depois rode a célula (▶).
variavel = "n_palavras"  #@param ["n_palavras", "n_caracteres", "tem_negacao", "tem_adversativa", "fala_de_recomendar", "fala_de_entrega", "so_maiuscula", "tem_exclamacao"]
tipo_de_grafico = "Histograma sobreposto"  #@param ["Histograma sobreposto", "Boxplot", "Violino", "Barras de proporcao", "Termos mais frequentes por classe"]

# Em texto, além das variáveis numéricas construídas, é possível analisar
# diretamente os termos. Cada gráfico responde a uma pergunta diferente.
dados["n_palavras"]   = dados.mensagem.str.split().str.len()
dados["n_caracteres"] = dados.mensagem.str.len()
dados["tem_negacao"]  = dados.mensagem.str.contains(
    r"\bn[aã]o\b|\bnem\b|nunca|jamais", case=False).astype(int)
dados["tem_adversativa"] = dados.mensagem.str.contains(
    r"\bmas\b|por[eé]m|contudo|entretanto|apesar|s[oó] que", case=False).astype(int)
dados["fala_de_recomendar"] = dados.mensagem.str.contains(r"recomend", case=False).astype(int)
dados["fala_de_entrega"] = dados.mensagem.str.contains(
    r"entrega|prazo|chegou|frete|correio|demor|transportadora", case=False).astype(int)
dados["so_maiuscula"] = dados.mensagem.str.isupper().astype(int)
dados["tem_exclamacao"] = dados.mensagem.str.contains(r"!", regex=True).astype(int)

CORES_C = ["#E6177A", "#3DCB55"]   # 0 = insatisfeito, 1 = satisfeito
serie = dados[variavel]
eh_binaria = serie.nunique() <= 3
rot_leg = ["Insatisfeito", "Satisfeito"]
# A cauda é longuíssima (há avaliações de centenas de palavras). Sem recorte,
# o gráfico vira uma barra colada no zero e não mostra nada.
limite = None if eh_binaria else float(serie.quantile(0.99))
vista = serie if eh_binaria else serie.clip(upper=limite)
grupos = [vista[dados.rotulo == k] for k in [0, 1]]

INCOMPAT = {
    ("Histograma sobreposto", True): "a variável só assume 0 ou 1; o histograma vira duas barras e esconde a informação",
    ("Violino", True): "o violino estima densidade contínua, que não existe para uma variável binária",
    ("Boxplot", True): "quartis de uma variável binária não são informativos",
    ("Barras de proporcao", False): "proporção por categoria exige poucos valores distintos, e aqui há centenas",
}
aviso = INCOMPAT.get((tipo_de_grafico, eh_binaria))
print(f"Variável: {variavel}  |  valores distintos: {serie.nunique()}  |  "
      f"tipo: {'BINÁRIA' if eh_binaria else 'CONTÍNUA'}")
if aviso:
    print(f"\n[!] ATENÇÃO: {tipo_de_grafico} não é a melhor escolha aqui, porque {aviso}.")
    print(f"    Prefira: {'Barras de proporcao' if eh_binaria else 'Histograma, Boxplot ou Violino'}.")
    print("    O gráfico será exibido assim mesmo, para vocês verem o problema.\n")

if tipo_de_grafico == "Termos mais frequentes por classe":
    from sklearn.feature_extraction.text import CountVectorizer
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax_, k, cor, titulo in zip(axes, [0, 1], CORES_C, rot_leg):
        cv = CountVectorizer(max_features=12, min_df=2)
        freq = np.asarray(cv.fit_transform(
            dados[dados.rotulo == k].mensagem).sum(axis=0)).ravel()
        termos = np.array(cv.get_feature_names_out())
        ordem = np.argsort(freq)
        ax_.barh(termos[ordem], freq[ordem], color=cor, edgecolor="#1F2A44")
        ax_.set_title(f"Termos mais frequentes: {titulo}", fontweight="bold")
        ax_.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); plt.show()
    print("Reparem que quase todos os termos aparecem nos DOIS lados. Frequência alta")
    print("não significa poder discriminante: o que separa as classes é o termo que")
    print("é frequente em uma e raro na outra, e é exatamente isso que o TF-IDF mede.")
else:
    fig, ax = plt.subplots(figsize=(8.6, 4.4))
    if tipo_de_grafico == "Histograma sobreposto":
        bins = np.histogram_bin_edges(vista, bins=28)
        for g, c, r in zip(grupos, CORES_C, rot_leg):
            ax.hist(g, bins=bins, alpha=.6, label=r, color=c, edgecolor="white")
        ax.legend(); ax.set_xlabel(variavel); ax.set_ylabel("quantidade")
    elif tipo_de_grafico == "Boxplot":
        bp = ax.boxplot(grupos, patch_artist=True, tick_labels=rot_leg,
                        medianprops=dict(color="black", linewidth=2))
        for p, c in zip(bp["boxes"], CORES_C): p.set_facecolor(c); p.set_alpha(.8)
        ax.set_ylabel(variavel)
    elif tipo_de_grafico == "Violino":
        vp = ax.violinplot(grupos, showmedians=True)
        for corpo, c in zip(vp["bodies"], CORES_C): corpo.set_facecolor(c); corpo.set_alpha(.75)
        ax.set_xticks([1, 2]); ax.set_xticklabels(rot_leg); ax.set_ylabel(variavel)
    else:
        tab = pd.crosstab(dados[variavel], dados.rotulo, normalize="index") * 100
        base = np.zeros(len(tab))
        for k, c in zip([0, 1], CORES_C):
            vals = tab[k].values if k in tab.columns else np.zeros(len(tab))
            ax.bar(tab.index.astype(str), vals, bottom=base, color=c,
                   label=rot_leg[k], edgecolor="white")
            base += vals
        ax.set_xlabel(variavel); ax.set_ylabel("% dentro de cada valor"); ax.legend()
    ax.set_title(f"'{variavel}' por classe", fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); plt.show()
    if limite is not None:
        print(f"\n(Eixo recortado no percentil 99, em {limite:.0f}. A cauda vai bem além disso.)")
    i_, s_ = dados[dados.rotulo==0][variavel].mean(), dados[dados.rotulo==1][variavel].mean()
    print(f"\nMédia de '{variavel}':  insatisfeito {i_:.3f}  |  satisfeito {s_:.3f}  "
          f"(diferença {i_-s_:+.3f})")
    print("   Diferenças pequenas indicam marcadores compartilhados entre as classes.")
    print("   São eles que tornam o problema não trivial: um filtro baseado apenas em")
    print("   caixa alta ou menção a prazo produziria muitos falsos positivos.")
    print("   Diferenças grandes são pistas, mas nenhuma delas sozinha resolve a tarefa.")

> **Vera:** *"Vocês identificaram os marcadores. Agora precisam de um método que os
> combine com pesos ajustados a partir de evidência, e não de intuição."*

<img src="https://lh3.googleusercontent.com/d/16tR_5M-HDAjeTC22mIoH97BBuhImZAAf=w1600" width="100%"
     alt="Três jovens de costas observam uma avaliação com estrelas se desmanchar em palavras soltas que atravessam a sala e se acendem, poucas de cada vez, numa grade enorme e quase toda vazia em ciano e magenta.">


---
# ⚙️ BLOCO 2 · CONFIGURAR
## *As decisões que definem o desempenho*

### Como os dados estão organizados

| Conjunto | Tamanho | Vocês têm os rótulos? | Para que serve |
|:---|:---|:---|:---|
| **Público** | 109.982 avaliações | Sim | Treinar o modelo e validar as escolhas |
| **Teste** | **900 (fixo)** | Não | Comparação oficial entre todas as equipes |

O controle abaixo define a fatia do conjunto público reservada para **validação**.
O conjunto de teste tem tamanho fixo e idêntico para todas as equipes, e a submissão
precisa ter exatamente 900 linhas.

> Com cento e dez mil documentos, o vocabulário completo tem dezenas de milhares de termos.
> Os valores iniciais dos controles abaixo são **deliberadamente modestos**: eles produzem
> um modelo que funciona e que está longe do que o dado permite. Mexer neles é parte do
> desafio.


In [ ]:
#@title 🎛️ Parâmetros do experimento { display-mode: "form" }
#@markdown Ajuste os controles e rode a célula (▶) para registrar a configuração.
#@markdown Os valores iniciais são modestos de propósito: mexer neles é parte do desafio.
tamanho_da_validacao = 25  #@param {type:"slider", min:10, max:40, step:5}
max_palavras = 400  #@param {type:"slider", min:200, max:50000, step:200}
ngramas = "so 1 palavra"  #@param ["so 1 palavra", "1 a 2 palavras", "1 a 3 palavras"]
min_df = 3  #@param {type:"slider", min:1, max:20, step:1}
modelo_escolhido = "Naive Bayes"  #@param ["Regressao Logistica", "SVM Linear", "Naive Bayes"]
forca_C = 1.0  #@param {type:"slider", min:0.01, max:20, step:0.01}

FAIXA = {"so 1 palavra": (1,1), "1 a 2 palavras": (1,2), "1 a 3 palavras": (1,3)}
faixa_ngrama = FAIXA[ngramas]

print("PARÂMETROS DEFINIDOS")
print(f"   Validação .......... {tamanho_da_validacao}% do conjunto público  "
      f"({int(len(dados)*tamanho_da_validacao/100)} avaliações)")
print(f"   Vocabulário ........ até {max_palavras} termos")
print(f"   N-gramas ........... {ngramas}  {faixa_ngrama}")
print(f"   min_df ............. {min_df}")
print(f"   Classificador ...... {modelo_escolhido}")
print(f"   Força C ............ {forca_C}")
print(f"   Teste .............. {N_TESTE} avaliações  [FIXO para todas as equipes]")
if faixa_ngrama == (1, 1):
    print("\n[!] Com 'so 1 palavra', 'recomendo' e 'não recomendo' compartilham o mesmo")
    print("    termo de maior peso. Vale testar o que acontece ao incluir bigramas.")
print("\nConfiguração registrada. O Bloco 3 tem várias lacunas: leiam o texto antes.")

### As decisões e seus efeitos

**Dimensão do vocabulário.** Define quantos termos compõem o espaço de representação. Um
vocabulário maior preserva mais informação, mas aumenta a dimensionalidade e o risco de
sobreajuste. Um vocabulário pequeno demais descarta sinal: com cento e dez mil avaliações,
o mesmo juízo aparece redigido de centenas de maneiras diferentes, e algumas centenas de
termos capturam apenas as mais óbvias.

**Faixa de n-gramas.** Um n-grama é uma sequência contígua de n termos. Unigramas descartam
a ordem (modelo de *saco de palavras*). Incluir bigramas recupera parte da informação
sintática, e neste corpus isso é decisivo: com unigramas, `recomendo` e `não recomendo`
compartilham o termo de maior peso, e a negação simplesmente desaparece. `não recomendo` só
existe como unidade a partir de `(1, 2)`.

**Frequência documental mínima (`min_df`).** Descarta termos presentes em menos de um
número mínimo de documentos, evitando que o modelo associe termos raríssimos, incluindo
erros de digitação de uma pessoa só, a uma classe por acidente.

**Desequilíbrio entre as classes.** O acervo tem cerca de 71% de satisfeitos contra 29% de
insatisfeitos, e a classe minoritária é justamente a crítica. Existe um parâmetro que trata
disso diretamente nas tabelas abaixo. Procurem por ele.

---

### 📗 Referência técnica: nome exato e parâmetros de cada classe

**Vocês vão precisar desta tabela no Bloco 3.**

> **Por que isso importa.** No Bloco 3, apenas alguns parâmetros aparecem preenchidos.
> Os demais assumem silenciosamente o valor padrão. Completar só o nome da classe produz
> um resultado correto e mediano. Estudar esta tabela e **acrescentar parâmetros à chamada**
> produz resultados melhores. A diferença é intencional.

#### 🔤 Vetorizador TF-IDF

```python
from sklearn.feature_extraction.text import TfidfVectorizer
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `max_features` | `None` | Limita o vocabulário aos termos mais frequentes. `None` mantém todos |
| `ngram_range` | `(1, 1)` | Faixa de n-gramas. `(1, 2)` inclui bigramas |
| `min_df` | `1` | Frequência documental mínima. Aceita inteiro ou fração |
| `max_df` | `1.0` | Descarta termos presentes em mais que essa fração dos documentos |
| `lowercase` | `True` | Converte tudo para minúsculas antes de tokenizar |
| `strip_accents` | `None` | `'unicode'` remove acentos e unifica grafias |
| `stop_words` | `None` | Lista de palavras a descartar |
| `sublinear_tf` | `False` | **`True` aplica escala logarítmica à frequência e costuma ajudar em texto** |
| `use_idf` | `True` | Se aplica o fator IDF |
| `smooth_idf` | `True` | Suaviza o IDF para evitar divisão por zero |
| `norm` | `'l2'` | Normalização de cada linha |
| `binary` | `False` | `True` considera apenas presença, ignorando contagem |
| `token_pattern` | `r"(?u)\b\w\w+\b"` | Regex do que conta como termo. O padrão **descarta palavras de uma letra** |

> ⚠️ Atenção ao `stop_words` neste desafio. Listas prontas de palavras vazias para o
> português costumam incluir `não`, `nem` e `mas` — exatamente os termos que carregam a
> polaridade aqui. Descartá-los é uma decisão que precisa ser medida, não presumida.

#### 📈 Regressão Logística

```python
from sklearn.linear_model import LogisticRegression
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `penalty` | `'l2'` | Regularização. Aceita `'l1'`, `'l2'`, `'elasticnet'`, `None` |
| `C` | `1.0` | Inverso da força da regularização. **Maior ajusta mais aos dados** |
| `solver` | `'lbfgs'` | Otimizador. `'liblinear'` e `'saga'` aceitam `'l1'` |
| `max_iter` | `100` | Iterações máximas. Em texto, frequentemente insuficiente |
| `class_weight` | `None` | `'balanced'` compensa desbalanceamento |
| `fit_intercept` | `True` | Se estima o termo constante |
| `random_state` | `None` | Semente |

#### ⚡ SVM Linear

```python
from sklearn.svm import LinearSVC
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `C` | `1.0` | Penalidade por erro. Maior ajusta mais ao treino |
| `penalty` | `'l2'` | Regularização |
| `loss` | `'squared_hinge'` | Função de perda. Aceita `'hinge'` |
| `class_weight` | `None` | `'balanced'` compensa desbalanceamento |
| `max_iter` | `1000` | Iterações máximas |
| `random_state` | `None` | Semente |

#### 🎲 Naive Bayes Multinomial

```python
from sklearn.naive_bayes import MultinomialNB
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `alpha` | `1.0` | Suavização de Laplace. Valores menores confiam mais nos dados observados |
| `fit_prior` | `True` | Se aprende a frequência das classes a partir dos dados |
| `class_prior` | `None` | Permite fixar as probabilidades a priori manualmente |

> 💡 **O `MultinomialNB` não aceita `class_weight`.** Se quiserem controlar o
> desbalanceamento com ele, o caminho é `class_prior`. Detalhes assim aparecem nas
> tabelas e não no código: é o tipo de leitura que separa as equipes.


---
# ✏️ BLOCO 3 · MODELAR
## *O erro mais clássico da Inteligência Artificial mora aqui*

### Dois estimadores encadeados

O pipeline agora tem duas etapas com responsabilidades distintas: um **transformador**, que
converte texto em vetores, e um **classificador**, que decide a classe. Cada um tem sua
interface.

**1. O TRADUTOR (o vetorizador TF-IDF)** — transforma texto em números.

| Comando | O que faz |
|:---|:---|
| `.fit_transform(textos)` | **APRENDE** o vocabulário E traduz o texto |
| `.transform(textos)` | **SÓ TRADUZ**, usando o vocabulário que já aprendeu antes |

**2. O CLASSIFICADOR (o modelo)** — decide satisfeito ou insatisfeito.

| Comando | O que faz |
|:---|:---|
| `.fit(X, y)` | **TREINA** o modelo |
| `.predict(X)` | **PREVÊ** |

### O ponto crítico: vazamento de dados

A questão que organiza esta etapa é a seguinte:

> **Por que o transformador usa `fit_transform` no treino e apenas `transform` no teste?**

O `fit` do vetorizador não é uma operação neutra. Ele estima parâmetros a partir dos dados:
define quais termos compõem o vocabulário e calcula o IDF de cada um, que depende das
frequências documentais **do corpus inteiro** ao qual foi ajustado.

Se o `fit` incluir o conjunto de teste, esses parâmetros passam a incorporar informação
sobre documentos que, por construção, deveriam ser desconhecidos no momento do treino. A
estimativa de desempenho resultante deixa de ser válida, porque mede o modelo em uma
condição que não se reproduz em operação real. Esse é o **vazamento de dados**
(*data leakage*).

O efeito é particularmente perigoso porque é silencioso: nenhum erro é lançado, o código
executa normalmente e a métrica final **melhora**. Sistemas com desempenho excelente em
validação e desempenho medíocre em produção frequentemente têm aqui a sua causa.

> **Princípio geral:** qualquer operação que **estime parâmetros a partir dos dados**
> (vocabulário, IDF, média, desvio-padrão, valores de imputação) deve ser ajustada
> exclusivamente sobre o conjunto de treino e apenas aplicada aos demais. A regra não vale
> só para texto: vale para todo pré-processamento em aprendizado de máquina.

### Sobre a estrutura condicional

O trecho abaixo precisa escolher um entre três classificadores conforme a opção marcada no
Bloco 2, usando uma cadeia `if` / `elif` / `else`. `else` não recebe condição, apenas os
dois-pontos.

> ⚠️ A célula não executa enquanto houver lacunas. Preencham primeiro, executem depois.


In [ ]:
# ✏️ SUA VEZ · Parte 1: separar os dados e construir as ferramentas
# Substituam cada ____ pelo que falta. A célula só executa quando não sobrar nenhum.
X = dados["mensagem"]
y = dados["rotulo"]

# LACUNA 1 ▸ 'stratify' precisa receber o vetor cuja proporção deve ser preservada
#            nas duas partes. É o texto das avaliações ou o vetor de rótulos?
X_treino, X_valida, y_treino, y_valida = train_test_split(
    X, y, test_size=tamanho_da_validacao/100, random_state=42, stratify=____)

# LACUNA 2 ▸ o nome exato da classe que transforma texto em TF-IDF está no Bloco 2.
vetorizador = ____(max_features=max_palavras, ngram_range=faixa_ngrama, min_df=min_df)

# LACUNAS 3 a 7 ▸ os nomes exatos dos classificadores estão na referência do Bloco 2.
#                 Faltam também as duas palavras que continuam a cadeia condicional.
if modelo_escolhido == "Regressao Logistica":
    modelo = ____(C=forca_C, max_iter=1000, random_state=42)

____ modelo_escolhido == "SVM Linear":
    modelo = ____(C=forca_C, random_state=42)

____:
    modelo = ____()

# ZONA LIVRE ▸ vocês podem acrescentar parâmetros às chamadas acima.
#   As tabelas do Bloco 2 listam os disponíveis e o valor que cada um assume quando
#   não é declarado. Reparem que existe um parâmetro, presente na Regressão Logística
#   e no SVM, que trata diretamente do desequilíbrio entre as classes visto no Bloco 1.
#   Alterem um de cada vez e registrem o efeito sobre o F1-macro e sobre o recall da
#   classe crítica, no Bloco 4. Os dois nem sempre andam juntos.

print(f"Treino:    {len(X_treino)} avaliações")
print(f"Validação: {len(X_valida)} avaliações")
print(f"Vetorizador: {type(vetorizador).__name__}  |  Classificador: {type(modelo).__name__}")


In [ ]:
# ✏️ SUA VEZ · Parte 2: vetorizar, treinar e prever
# Atenção: as lacunas 8 e 9 são as duas de nome parecido. Releiam o Bloco 3 antes.

# LACUNA 8 ▸ no TREINO, o vetorizador precisa APRENDER o vocabulário e o IDF, e
#            também devolver a matriz transformada. Qual método faz as duas coisas?
X_treino_vec = vetorizador.____(X_treino)

# LACUNA 9 ▸ na VALIDAÇÃO, ele deve apenas APLICAR o que já aprendeu, sem reestimar
#            nada. Qual método faz só isso?
X_valida_vec = vetorizador.____(X_valida)

# LACUNA 10 ▸ qual método ajusta os coeficientes do classificador ao conjunto de treino?
modelo.____(X_treino_vec, y_treino)

# LACUNA 11 ▸ qual método infere as classes de avaliações que o modelo nunca viu?
previsoes = modelo.____(X_valida_vec)

print(f"Vocabulário aprendido: {len(vetorizador.vocabulary_)} termos")
print(f"Previsões geradas: {len(previsoes)}")


In [ ]:
#@title 🔎 Verificação das lacunas (inclusive vazamento) { display-mode: "form" }
erros = []
try:
    # Um split estratificado reproduz a composição de cada classe com erro de no máximo
    # um exemplo. Sem stratify, o desvio típico neste acervo é da ordem de uma centena.
    esperado = np.bincount(y, minlength=2) * len(y_treino) / len(y)
    desvio = np.abs(np.bincount(y_treino, minlength=2) - esperado).max()
    if desvio > 5:
        erros.append(f"LACUNA 1: o treino ficou {desvio:.0f} exemplos fora da composição "
                     "que um split estratificado produziria. O 'stratify' não recebeu o "
                     "vetor cuja proporção deveria ser preservada.")
except NameError:
    erros.append("LACUNA 1: X_treino/y_treino não existem.")
try:
    if type(vetorizador).__name__ != "TfidfVectorizer":
        erros.append(f"LACUNA 2: esperava TfidfVectorizer, obteve {type(vetorizador).__name__}.")
except NameError:
    erros.append("LACUNA 2: a variável 'vetorizador' não existe.")
try:
    esp = {"Regressao Logistica":"LogisticRegression","SVM Linear":"LinearSVC",
           "Naive Bayes":"MultinomialNB"}[modelo_escolhido]
    if type(modelo).__name__ != esp:
        erros.append(f"LACUNAS 3 a 7: para '{modelo_escolhido}' era esperada a classe {esp}, "
                     f"mas foi construída {type(modelo).__name__}. Revise os nomes e a "
                     f"cadeia if / elif / else.")
except NameError:
    erros.append("LACUNAS 3 a 7: a variável 'modelo' não existe.")

vocab_atual = set()
try: vocab_atual = set(vetorizador.vocabulary_.keys())
except Exception:
    erros.append("LACUNA 8: o vetorizador não tem parâmetros estimados. Qual método "
                 "estima e aplica a transformação simultaneamente?")
if vocab_atual:
    # Referência: um vetorizador com os MESMOS parâmetros da equipe, ajustado só no treino.
    _v = TfidfVectorizer(**vetorizador.get_params())
    _v.fit(X_treino)
    if vocab_atual != set(_v.vocabulary_.keys()):
        erros.append("LACUNA 9: VAZAMENTO DE DADOS DETECTADO. O vocabulário estimado difere "
                     "do que se obteria ajustando apenas sobre o treino, o que indica que a "
                     "validação participou da estimação. O método aplicado reestima em vez "
                     "de apenas aplicar. Note que nenhuma exceção foi lançada e a métrica "
                     "ficaria artificialmente melhor: é assim que esse erro passa despercebido.")
try:
    if X_valida_vec.shape[1] != X_treino_vec.shape[1]:
        erros.append("LACUNAS 8 e 9: treino e validação ficaram com números de colunas "
                     "diferentes. Os dois precisam usar o MESMO vocabulário.")
except NameError:
    erros.append("LACUNAS 8 ou 9: faltou criar X_treino_vec ou X_valida_vec.")
try:
    if not (hasattr(modelo, "classes_") or hasattr(modelo, "coef_")):
        erros.append("LACUNA 10: o classificador não foi ajustado.")
except NameError: pass
try:
    if len(previsoes) != len(X_valida):
        erros.append(f"LACUNA 11: {len(previsoes)} previsões para {len(X_valida)} avaliações.")
except NameError:
    erros.append("LACUNA 11: a variável 'previsoes' não existe.")

if erros:
    print("VERIFICAÇÃO REPROVADA. Pontos a revisar:\n")
    for e in erros: print("   [!] " + e + "\n")
else:
    print("VERIFICAÇÃO APROVADA\n")
    print("   Não houve vazamento: o vocabulário foi estimado apenas sobre o treino.")
    print("   A avaliação a seguir é uma estimativa honesta do desempenho real.")
    pv = {k: v for k, v in vetorizador.get_params().items()
          if k in ("sublinear_tf","strip_accents","binary","max_df","stop_words")
          and v not in (False, None, 1.0)}
    pm = {k: v for k, v in modelo.get_params().items()
          if k in ("class_weight","alpha","penalty","loss") and v not in (None, 1.0, "l2", "squared_hinge")}
    if pv or pm:
        print(f"\n   Parâmetros adicionais na ZONA LIVRE: {{**pv, **pm}}")
    else:
        print("\n   Nenhum parâmetro adicional foi declarado: tudo nos valores padrão.")
        print("   É válido, mas as tabelas do Bloco 2 dão margem real de melhoria.")


---
# 📈 BLOCO 4 · AVALIAR
## *Quanto custa cada tipo de erro?*

> **Vera:** *"Voltem à assimetria que mencionei no início. Ela precisa aparecer na forma
> como vocês avaliam, e não apenas no texto do relatório."*

Um classificador binário comete dois tipos de erro, e a distinção entre eles é operacional:

- **Marcar como insatisfeito quem estava satisfeito:** custo baixo. Um contato
  desnecessário e uma posição a menos na fila de atendimento.
- **Deixar passar um insatisfeito:** custo alto. Ninguém procura esse cliente. A próxima
  notícia costuma ser uma reclamação pública, um pedido de estorno ou a perda da conta.

A **classe crítica é a 0, insatisfeito** — e ela é a minoritária, com cerca de 29% do
acervo. Isso tem duas consequências diretas:

1. **A acurácia é insuficiente.** Um classificador degenerado que respondesse "satisfeito"
   para tudo já acertaria 71% sem aprender nada, executando exatamente o oposto da tarefa.
   Por isso a métrica oficial é o **F1-macro**, que calcula o F1 de cada classe e tira a
   média simples: a classe pequena pesa igual à grande.
2. **Vale acompanhar a revocação da classe 0 em separado**, porque é ela que traduz o custo
   que a Vera descreveu.

Duas métricas capturam a distinção sobre a classe crítica:

- **Precisão** = das avaliações sinalizadas como insatisfeitas, quantas realmente eram.
  Precisão baixa significa fila de atendimento entupida de gente que não precisava ser
  contatada.
- **Revocação** (*recall*) = dos insatisfeitos existentes, quantos o sistema capturou.
  Revocação baixa significa clientes irritados que ninguém procurou.

As duas se opõem. Tornar o classificador mais sensível eleva a revocação e reduz a precisão;
torná-lo mais conservador faz o inverso. Esse **compromisso entre precisão e revocação** não
tem solução técnica: a posição escolhida depende dos custos relativos no domínio de
aplicação, e é uma decisão de projeto, não de otimização.


In [ ]:
#@title 🏆 O resultado da escuta { display-mode: "form" }
acuracia = accuracy_score(y_valida, previsoes)
f1 = f1_score(y_valida, previsoes, average="macro")
recall_insatisfeito = recall_score(y_valida, previsoes, pos_label=CLASSE_CRITICA)
precisao_insatisfeito = precision_score(y_valida, previsoes, pos_label=CLASSE_CRITICA)

print("="*62)
print("            DESEMPENHO NO CONJUNTO DE VALIDAÇÃO")
print("="*62)
print(f"  Acurácia (acertos no geral) ................ {acuracia*100:5.1f}%")
print(f"  ⭐ F1-MACRO (métrica oficial) ............... {f1*100:5.1f}%")
print(f"  🔴 RECALL DO INSATISFEITO (o que mais importa) {recall_insatisfeito*100:5.1f}%")
print(f"     Precisão do insatisfeito ................ {precisao_insatisfeito*100:5.1f}%")
print("="*62)
print(f"  Referência: responder 'satisfeito' para tudo daria "
      f"{(y_valida==1).mean()*100:.1f}% de acurácia e F1-macro de 41%.")
if   f1 >= .94: print("\n  Desempenho alto. Avaliem a estabilidade e o risco de sobreajuste.")
elif f1 >= .91: print("\n  Desempenho sólido. Examinem a revocação da classe crítica.")
elif f1 >= .86: print("\n  Desempenho intermediário. Revisem n-gramas e vocabulário.")
else:           print("\n  Desempenho baixo. A representação atual não separa as classes.")

cm = confusion_matrix(y_valida, previsoes)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=["Insatisfeito","Satisfeito"]).plot(
    ax=ax, cmap="viridis", colorbar=False)
ax.set_title("Onde a escuta acertou e onde falhou", fontweight="bold")
ax.set_xlabel("O que o modelo DISSE"); ax.set_ylabel("O que REALMENTE era")
plt.tight_layout(); plt.show()

print("\nDECOMPOSIÇÃO DOS ERROS:\n")
print(f"   INSATISFEITOS NÃO DETECTADOS ................ {cm[0,1]}")
print("      Erro de custo alto: esse cliente não entra na fila e ninguém o procura.")
print(f"\n   SATISFEITOS SINALIZADOS SEM NECESSIDADE ..... {cm[1,0]}")
print("      Erro de custo baixo, mas não nulo: ocupa a fila de atendimento e, em")
print("      excesso, faz a ouvidoria deixar de confiar na triagem automática.")
print("\n   Calculem a razão entre os dois e avaliem se ela corresponde à assimetria")
print("   de custo do domínio. Se não corresponder, o ponto de operação está mal")
print("   calibrado, ainda que o F1 agregado pareça satisfatório.\n")
print(classification_report(y_valida, previsoes,
      target_names=["Insatisfeito","Satisfeito"], digits=3))


In [ ]:
#@title 🔤 As palavras que entregam um cliente irritado { display-mode: "form" }
# A parte mais divertida: descobrir o que a IA aprendeu sozinha.
# Em problema binário, coef_[0] é o peso a favor da classe 1 (satisfeito):
# coeficiente negativo puxa para a classe 0 (insatisfeito).
if hasattr(modelo, "coef_"):
    termos = np.array(vetorizador.get_feature_names_out())
    pesos = modelo.coef_[0]
    ordem = np.argsort(pesos)
    top_insatisfeito = termos[ordem[:12]]
    val_insatisfeito = np.abs(pesos[ordem[:12]])
    top_satisfeito   = termos[ordem[-12:]][::-1]
    val_satisfeito   = pesos[ordem[-12:]][::-1]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].barh(range(12), val_insatisfeito[::-1], color="#E6177A")
    axes[0].set_yticks(range(12)); axes[0].set_yticklabels(top_insatisfeito[::-1])
    axes[0].set_title("🔴 Termos que mais indicam INSATISFEITO", fontweight="bold")
    axes[1].barh(range(12), val_satisfeito[::-1], color="#3DCB55")
    axes[1].set_yticks(range(12)); axes[1].set_yticklabels(top_satisfeito[::-1])
    axes[1].set_title("🟢 Termos que mais indicam SATISFEITO", fontweight="bold")
    for a in axes: a.spines[["top","right"]].set_visible(False)
    plt.tight_layout(); plt.show()

    print("TERMOS DE MAIOR PESO PARA A CLASSE CRÍTICA (insatisfeito):")
    for i, t in enumerate(top_insatisfeito[:5], 1): print(f"   {i}. \"{t}\"")
    if faixa_ngrama[1] == 1:
        print("\n   Todos são palavras isoladas, porque o vocabulário atual só tem unigramas.")
        print("   Voltem ao Bloco 2, liguem os bigramas e vejam quais expressões aparecem.")
    else:
        _bi = [t for t in top_insatisfeito if " " in t]
        print(f"\n   Expressões de mais de uma palavra entre os 12 primeiros: {len(_bi)}")
        if _bi: print(f"   {', '.join(_bi[:4])}")
    print("\nEstes pesos foram estimados a partir dos dados, sem regra escrita à mão.")
    print("Comparem com as hipóteses formuladas no Bloco 1: o modelo confirmou,")
    print("refutou, ou encontrou marcadores que vocês não haviam previsto?")
    print("\nRessalva: coeficiente alto indica associação no ACERVO DE TREINO. Termos ligados")
    print("a um produto ou a uma campanha específica discriminam bem hoje e deixam de")
    print("discriminar quando o catálogo e a forma de escrever mudam.")
else:
    print("ℹ️ O Naive Bayes não expõe pesos da mesma forma.")
    print("   Voltem ao Bloco 2 e escolham Regressão Logística ou SVM Linear")
    print("   para ver quais palavras entregam um cliente irritado!")


In [ ]:
#@title 🧪 Testem a escuta com uma avaliação escrita por vocês! { display-mode: "form" }
#@markdown Escrevam uma avaliação qualquer e rodem a célula (▶).
sua_mensagem = "Demorou muito pra chegar, mas o produto \u00e9 excelente. Recomendo."  #@param {type:"string"}

vec = vetorizador.transform([sua_mensagem])
palpite = modelo.predict(vec)[0]

print("="*62)
print(f'  AVALIAÇÃO: "{sua_mensagem}"')
print("="*62)
print(f"\n  👂  A ESCUTA DIZ: {NOME_CLASSE[palpite]}\n")
if hasattr(modelo, "predict_proba"):
    p = modelo.predict_proba(vec)[0][CLASSE_CRITICA]
    print(f"      Confiança de ser um cliente insatisfeito: {p*100:.1f}%")
    barra = "█" * int(p*40) + "░" * (40 - int(p*40))
    print(f"      [{barra}]")
print("\n  Testem os casos difíceis: uma avaliação que elogia o produto e reclama do")
print("  prazo, uma que use ironia ('parabéns pela agilidade, só levou um mês'), e uma")
print("  reclamação sem nenhuma palavra negativa óbvia. Onde o modelo erra, vocês")
print("  encontraram o limite do saco de palavras — e material para o relatório.")

---
# 🤔 PARADA OBRIGATÓRIA: o outro lado da escuta

> **Vera:** *"Quatro questões antes de vocês fecharem. Nenhuma é sobre desempenho."*
>
> *"Comecem por esta avaliação, que é real: 'Produto perfeito, chegou certinho. Só não dou
> cinco estrelas porque o preço subiu depois que comprei.' O cliente deu duas estrelas. No
> acervo de vocês, ele é um insatisfeito. Vocês treinaram um modelo para reproduzir esse
> julgamento. A pergunta é o que exatamente ele está aprendendo."*

### Questões para a equipe deliberar

1. **A procedência do rótulo.** O rótulo não foi atribuído por um especialista: ele vem da
   nota em estrelas dada pelo próprio cliente, e essa nota mistura produto, entrega, preço
   e humor do dia. O modelo, portanto, não aprende "satisfação": aprende **a relação entre o
   texto e a estrela**. Que erro sistemático isso injeta? E por que a consistência de quem
   rotulou funciona como um **teto** para o desempenho de qualquer modelo treinado ali?

2. **Viés linguístico e distribuição de erro.** O modelo foi ajustado sobre uma amostra
   específica. Escrita fora do padrão que predomina no acervo, incluindo variedades
   regionais, escrita com desvios ortográficos, avaliações de três palavras ou texto em
   caixa alta, tende a receber tratamento pior. O problema não é a taxa de erro global, e
   sim sua **distribuição desigual entre grupos**: um sistema pode ter bom desempenho
   agregado e desempenho ruim justamente sobre a população menos representada nos dados. Que
   procedimento vocês adotariam para verificar se isso ocorre no modelo de vocês? Pensem em
   como seria preciso segmentar a avaliação.

3. **O mesmo modelo, dois usos.** O uso declarado pela Vera é **priorizar atendimento**:
   quem está insatisfeito é procurado primeiro. Mas a saída é um número, e o mesmo número
   permite **despriorizar avaliações negativas na página do produto**, empurrando-as para o
   fim da lista. O modelo é idêntico nos dois casos; o que muda é quem consome a saída e com
   que finalidade. O que impediria o segundo uso não é técnico — então o que seria? Registrem
   no relatório uma restrição de uso que vocês considerem inegociável.

4. **Degradação temporal.** O catálogo muda, entram produtos novos, o vocabulário de quem
   escreve muda, e períodos como a Black Friday alteram a mistura de reclamações. A
   distribuição dos dados se desloca ao longo do tempo, e isso se chama **deriva conceitual**
   (*concept drift*). Um modelo estático tem prazo de validade. Que indicadores permitiriam
   à Ouvidoria perceber que o modelo está degradando **antes** que os prejuízos apareçam?
   Note que a resposta não pode depender de saber o rótulo verdadeiro, já que ele não está
   disponível em tempo real.

<img src="https://lh3.googleusercontent.com/d/1hSKgQ7jW8atOnd9nP5EVIm-waHT0AZtW=w1600" width="100%"
     alt="A mesma pilha de avaliações, agora organizada em fila: as bolhas magenta ficaram no topo, com contorno brilhante, e uma atendente de headset retira a primeira delas com a mão. A decisão final continua sendo de uma pessoa.">


In [ ]:
#@title 📝 O relatório da sua equipe para a Ouvidoria { display-mode: "form" }
#@markdown Este texto vale nota e pode decidir o desempate.
nome_da_equipe = ""  #@param {type:"string"}
#@markdown **1. Explique o vazamento de dados: o que o fit estima e por que isso invalida a avaliação.**
resposta_1 = ""  #@param {type:"string"}
#@markdown **2. Quais termos mais entregam um cliente insatisfeito? Algum surpreendeu?**
resposta_2 = ""  #@param {type:"string"}
#@markdown **3. Que restrição de uso vocês consideram inegociável, e por quê?**
resposta_3 = ""  #@param {type:"string"}
#@markdown **4. Como a Ouvidoria perceberia a degradação do modelo sem ter o rótulo verdadeiro?**
resposta_4 = ""  #@param {type:"string"}

relatorio = {"equipe": nome_da_equipe, "modelo": modelo_escolhido,
             "max_palavras": max_palavras, "ngramas": ngramas, "min_df": min_df,
             "C": forca_C, "validacao_pct": tamanho_da_validacao,
             "f1_macro": round(f1, 4), "acuracia": round(acuracia, 4),
             "recall_insatisfeito": round(recall_insatisfeito, 4),
             "precisao_insatisfeito": round(precisao_insatisfeito, 4),
             "just_vazamento": resposta_1, "just_termos": resposta_2,
             "just_restricao_de_uso": resposta_3, "just_deriva": resposta_4}

faltando = [k for k in ["equipe","just_vazamento","just_termos",
                        "just_restricao_de_uso","just_deriva"]
            if not str(relatorio[k]).strip()]
if faltando:
    print("⚠️ Ainda faltam campos:", ", ".join(faltando))
else:
    print("Relatório completo. Prossigam para a submissão.")

In [ ]:
#@title 🚀 Gerar o arquivo de submissão { display-mode: "form" }
#@markdown Rode a célula (▶). Os dois arquivos são baixados automaticamente.
import json, pathlib

PASTA_SUBMISSAO = pathlib.Path("submissoes/nlp")
PASTA_SUBMISSAO.mkdir(parents=True, exist_ok=True)
if "relatorio" not in dir() or not isinstance(relatorio, dict):
    relatorio = {"equipe": "", "modelo": modelo_escolhido,
                 "max_palavras": max_palavras, "ngramas": ngramas, "min_df": min_df,
                 "C": forca_C, "validacao_pct": tamanho_da_validacao,
                 "f1_macro": round(f1, 4), "acuracia": round(acuracia, 4),
                 "recall_insatisfeito": round(recall_insatisfeito, 4),
                 "precisao_insatisfeito": round(precisao_insatisfeito, 4),
                 "just_vazamento": "", "just_termos": "",
                 "just_restricao_de_uso": "", "just_deriva": ""}

# Repare: aqui usamos apenas transform. Coerência com a Regra do Bloco 3.
X_final_vec = vetorizador.transform(mensagens_teste)
palpites = modelo.predict(X_final_vec)

submissao = pd.DataFrame({"id_mensagem": IDS_TESTE, "rotulo_previsto": palpites})
assert len(submissao) == N_TESTE, f"A submissão precisa ter {N_TESTE} linhas."
csv_saida = PASTA_SUBMISSAO / "submissao_nlp.csv"
json_saida = PASTA_SUBMISSAO / "relatorio_nlp.json"
submissao.to_csv(csv_saida, index=False)
with open(json_saida, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

print("="*60)
print("             ARQUIVOS PRONTOS PARA ENVIO")
print("="*60)
print(f"  submissao_nlp.csv   ({len(submissao)} linhas)")
print("  relatorio_nlp.json")
print("="*60)
_n_ins = int((palpites == CLASSE_CRITICA).sum())
print(f"\n  Encaminhados para atendimento: {_n_ins} das {len(palpites)} avaliações "
      f"({_n_ins/len(palpites)*100:.1f}%).")
print(f"  No acervo público, os insatisfeitos são {(dados.rotulo==0).mean()*100:.1f}%. "
      f"Uma diferença\n  grande entre os dois números merece explicação.")
display(submissao.head())

try:
    from google.colab import files
    files.download(str(csv_saida))
    files.download(str(json_saida))
    print("\n  Download iniciado. Enviem os dois arquivos na plataforma.")
except ImportError:
    print(f"\n  Fora do Colab: os arquivos estão em {PASTA_SUBMISSAO.resolve()}")

---
# 🌟 FIM DA OPERAÇÃO ESCUTA

> **Vera:** *"Rodei o classificador de vocês sobre a fila de ontem. O resultado ficou dentro
> do que vocês reportaram, o que já é significativo: modelo que se comporta em produção como
> se comportou em validação é menos comum do que deveria."*
>
> *"O que me interessou mais, porém, foram duas coisas que eu não tinha pedido. A primeira
> foi vocês escreverem, com todas as letras, que o rótulo é a estrela do cliente e não a
> verdade sobre a satisfação dele. A segunda foi a restrição de uso: vocês entregaram o
> modelo e entregaram junto o que não se deve fazer com ele."*
>
> *"Quem entrega um classificador sem plano de monitoramento entrega metade do sistema. E
> quem entrega sem os limites de uso entrega um problema. É um trabalho de nível
> profissional. Obrigada."*

---

### Competências desenvolvidas neste desafio

- Representação vetorial de texto e a formulação do TF-IDF
- Matrizes esparsas e viabilidade computacional de vocabulários grandes
- N-gramas, o modelo de saco de palavras e o tratamento da negação
- Procedência do rótulo: supervisão derivada, ruído de anotação e teto de desempenho
- Vazamento de dados: mecanismo, invisibilidade e efeito sobre a validação
- O princípio geral do ajuste restrito ao treino em todo pré-processamento
- Classes desbalanceadas e por que o F1-macro substitui a acurácia
- Precisão, revocação e o compromisso entre as duas
- Assimetria de custo entre os dois tipos de erro
- Interpretação de coeficientes e confronto com hipóteses prévias
- Viés linguístico e distribuição desigual do erro entre grupos
- Restrição de uso: o mesmo modelo servindo a finalidades distintas
- Deriva conceitual e monitoramento sem rótulo disponível

**Otimização.** O objetivo não é maximizar o F1 isoladamente, e sim encontrar um ponto de
operação em que a revocação da classe crítica seja compatível com a assimetria de custo do
domínio, sem que a precisão caia a ponto de entupir a fila de atendimento e tornar a triagem
automática inútil na prática.
